# 17 · APIs & HTTP

A huge share of ingestion is pulling data from **REST APIs**. This notebook
covers the request/response model, status codes, JSON payloads, pagination,
and retry/backoff — using the `requests` library.

> To keep the course fully offline and deterministic, we run against a small
> **simulated API** defined in the next cell. The `requests` code you'd write
> against a real endpoint is shown in the markdown so you learn the real shape.

## The HTTP model

A client sends a **request** (method + URL + headers) and gets a **response**
(status code + headers + body). For data work the method is usually `GET` and
the body is JSON. Status codes tell you what happened:

- `2xx` success (200 OK)
- `4xx` client error (401 unauthorized, 404 not found, 429 too many requests)
- `5xx` server error (500, 503) — usually worth **retrying**

With `requests` the real code is:

```python
import requests
resp = requests.get('https://api.example.com/orders',
                    params={'page': 1}, headers={'Authorization': 'Bearer TOKEN'},
                    timeout=10)
resp.raise_for_status()      # raise on 4xx/5xx
data = resp.json()           # parse JSON body -> dict
```

In [ ]:
# A tiny in-memory API so this notebook runs offline.
import math

_ALL = [{'id': i, 'amount': (i * 13) % 200} for i in range(1, 48)]
PAGE_SIZE = 10

def fake_get(page):
    '''Mimics GET /orders?page=N returning a JSON body + status code.'''
    start = (page - 1) * PAGE_SIZE
    chunk = _ALL[start:start + PAGE_SIZE]
    total_pages = math.ceil(len(_ALL) / PAGE_SIZE)
    status = 200 if chunk else 404
    return status, {'page': page, 'total_pages': total_pages, 'results': chunk}

status, body = fake_get(1)
print('status:', status)
print('page 1 of', body['total_pages'], '- got', len(body['results']), 'records')
print('first:', body['results'][0])

## Pagination: fetch every page

APIs cap how many records they return per call. You loop, incrementing the page,
until the response tells you you're done. This `while` loop mirrors real
`requests`-based ingestion exactly.

In [ ]:
all_records = []
page = 1
while True:
    status, body = fake_get(page)
    if status == 404 or not body['results']:
        break
    all_records.extend(body['results'])
    if page >= body['total_pages']:
        break
    page += 1

print('fetched', len(all_records), 'records across', page, 'pages')
print('total amount:', sum(r['amount'] for r in all_records))

## Retries with exponential backoff

Transient `5xx` errors and rate limits (`429`) are normal. Robust clients
**retry** a few times, waiting longer each attempt (backoff), then give up.
Below is a reusable retry wrapper — the same logic you'd wrap around
`requests.get`.

In [ ]:
import time

_attempts = {'n': 0}
def flaky_get():
    '''Fails twice with 503, then succeeds — to demonstrate retries.'''
    _attempts['n'] += 1
    if _attempts['n'] < 3:
        return 503, None
    return 200, {'ok': True}

def get_with_retry(fn, max_tries=5, base_delay=0.01):
    for attempt in range(1, max_tries + 1):
        status, body = fn()
        if status == 200:
            print(f'success on attempt {attempt}')
            return body
        wait = base_delay * (2 ** (attempt - 1))    # 1x, 2x, 4x, ...
        print(f'attempt {attempt} got {status}; retrying in {wait:.3f}s')
        time.sleep(wait)
    raise RuntimeError('exhausted retries')

print(get_with_retry(flaky_get))

## Being a good client

Real ingestion also: sets a `timeout` on every request (never hang forever),
respects rate limits (sleep between calls or honor `Retry-After`), reuses a
`requests.Session` for connection pooling, and stores secrets in env vars — not
in code (the logging & config notebook). These habits keep pipelines reliable and polite.

In [ ]:
# Sketch of production-shaped ingestion (offline stand-in for the loop):
import time

def ingest_all():
    records, page = [], 1
    while True:
        status, body = fake_get(page)
        if status != 200 or not body['results']:
            break
        records.extend(body['results'])
        page += 1
        time.sleep(0.001)      # be polite between pages
    return records

print('ingested', len(ingest_all()), 'records')

### Recap

HTTP = request/response with status codes (`2xx/4xx/5xx`); parse JSON bodies to
dicts; paginate with a `while` loop until the API says stop; retry transient
failures with exponential backoff; always set timeouts and keep secrets in the
environment. Next: databases from Python.